In [ ]:
# ==========================================================
# STATISTICAL SIGNIFICANCE ANALYSIS FOR DIABETES DATASET
# ==========================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import LabelEncoder
from sklearn.feature_selection import f_classif

# ==========================================================
# LOAD DATASET
# ==========================================================

df = pd.read_csv('/content/Pediatrics.csv')

print("Dataset Shape:", df.shape)

# ==========================================================
# REMOVE DUPLICATES
# ==========================================================

df = df.drop_duplicates()

# ==========================================================
# HANDLE MISSING VALUES
# ==========================================================

df.fillna(df.mode().iloc[0], inplace=True)

# ==========================================================
# ENCODE CATEGORICAL VARIABLES
# ==========================================================

encoder = LabelEncoder()

for col in df.columns:
    if df[col].dtype == 'object':
        df[col] = encoder.fit_transform(df[col])

# ==========================================================
# DEFINE TARGET VARIABLE
# ==========================================================

target = 'Type of Diabetes'   # Change if your target column has another name

# ==========================================================
# SPLIT FEATURES AND TARGET
# ==========================================================

X = df.drop(columns=[target])
y = df[target]

# ==========================================================
# ANOVA F-TEST
# ==========================================================

f_scores, p_values = f_classif(X, y)

# ==========================================================
# CREATE RESULTS TABLE
# ==========================================================

results = pd.DataFrame({
    'Variable': X.columns,
    'F-Score': f_scores,
    'p-value': p_values
})

results['Significant (p < 0.05)'] = np.where(
    results['p-value'] < 0.05,
    'Yes',
    'No'
)

results = results.sort_values(
    by='p-value',
    ascending=True
)

results.reset_index(
    drop=True,
    inplace=True
)

results.insert(
    0,
    'Rank',
    range(1, len(results)+1)
)

# ==========================================================
# FORMAT P-VALUES
# ==========================================================

results['p-value'] = results['p-value'].apply(
    lambda x: "{:.2e}".format(x)
)

# ==========================================================
# DISPLAY RESULTS
# ==========================================================

print("\nSTATISTICAL SIGNIFICANCE ANALYSIS")
print(results)

# ==========================================================
# SAVE RESULTS
# ==========================================================

results.to_csv(
    'Statistical_Significance_Results.csv',
    index=False
)

# ==========================================================
# VISUALIZATION
# ==========================================================

plot_df = pd.DataFrame({
    'Variable': X.columns,
    'p-value': p_values
})

plot_df = plot_df.sort_values(
    by='p-value'
)

plt.figure(figsize=(10,6))

sns.barplot(
    data=plot_df,
    x='p-value',
    y='Variable'
)

plt.axvline(
    x=0.05,
    color='red',
    linestyle='--',
    label='Significance Threshold (0.05)'
)

plt.title(
    'Statistical Significance of Variables'
)

plt.xlabel('p-value')
plt.ylabel('Variables')

plt.legend()

plt.tight_layout()
plt.show()

print("\nAnalysis Completed Successfully.")